# Ingest: Card Transactions

Financial services demo data. Downloads the well-known Worldline/ULB
credit-card fraud dataset (284,807 European card transactions from
September 2013, PCA-anonymized features, public domain via OpenML — no
auth) and persists it as a managed Spark table.

Confirmed live (2026-08-07) columns: `Time` (seconds elapsed since the
first transaction in the dataset), `Amount`, `V1`-`V28` (anonymized PCA
features), `Class` (`"0"` = legitimate, `"1"` = fraud — 492 of 284,807,
~0.17%).

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and
`mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "payments"
TABLE_NAME = "card_transactions"
SOURCE_URL = "https://data.openml.org/datasets/0000/1597/dataset_1597.pq"
LOCAL_FILE = "/tmp/card_transactions.pq"

## Download source file

In [ ]:
import urllib.request

urllib.request.urlretrieve(SOURCE_URL, LOCAL_FILE)

## Load dataset

In [ ]:
df = spark.read.parquet(LOCAL_FILE)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'Card transaction data used for financial services fraud-detection demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_transactions FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)